# 02. 모델 A — ResNet18 스타일 분류 (CNN)
- **입력**: 224×224 RGB 이미지
- **백본**: ResNet18 (ImageNet 사전학습)
- **출력**: 스타일 24개 + 대분류 5개 (멀티태스크)
- **실제 데이터** : `C:\딥러닝데이터\KFashion\Training`
- **더미 모드** : 실제 데이터 없을 때 랜덤 텐서로 동작 확인

In [1]:
import os, sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as T
import torchvision.models as models

# KFashionDataset을 별도 파일에서 import (DataLoader 멀티프로세싱 필수)
sys.path.insert(0, str(Path('c:/Users/badah/OneDrive/문서/파이썬/딥러닝프로젝트')))
from kfashion_dataset import KFashionDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {device}')

디바이스: cuda


In [2]:
# K-Fashion 클래스 정의 (실제 데이터 기준)
STYLE_CLASSES = [
    '기타', '레트로', '로맨틱', '리조트', '매니시',
    '모던', '밀리터리', '섹시', '소피스트케이티드', '스트리트',
    '스포티', '아방가르드', '오리엔탈', '웨스턴', '젠더리스',
    '컨트리', '클래식', '키치', '톰보이', '펑크',
    '페미닌', '프레피', '히피', '힙합',
]
CATEGORY_CLASSES = ['상의', '하의', '아우터', '원피스', '점프수트']

NUM_STYLES     = len(STYLE_CLASSES)
NUM_CATEGORIES = len(CATEGORY_CLASSES)
IMG_SIZE       = 224
print(f'스타일 {NUM_STYLES}개 / 대분류 {NUM_CATEGORIES}개')

스타일 24개 / 대분류 5개


In [3]:
# 이미지 전처리 Transform
TRAIN_TRANSFORM = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
EVAL_TRANSFORM = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [4]:
class StyleCNN(nn.Module):
    """ResNet18 백본 + 스타일/대분류 멀티태스크 헤드"""
    def __init__(self, pretrained=True):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        self.style_head = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.4), nn.Linear(512, NUM_STYLES)
        )
        self.category_head = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.4), nn.Linear(512, NUM_CATEGORIES)
        )

    def forward(self, x):
        feat = self.backbone(x)
        return self.style_head(feat), self.category_head(feat)

In [5]:
# 데이터 로드
TRAIN_ROOT = r'C:\딥러닝데이터\KFashion\Training'
VAL_ROOT   = r'C:\딥러닝데이터\KFashion\Validation'
BATCH_SIZE = 128

if os.path.exists(TRAIN_ROOT):
    print('K-Fashion 실제 데이터 로드 중...')
    dataset = KFashionDataset(data_root=TRAIN_ROOT, transform=TRAIN_TRANSFORM)
    print(f'  → 학습 샘플: {len(dataset):,}개')

    if os.path.exists(VAL_ROOT):
        val_dataset = KFashionDataset(data_root=VAL_ROOT, transform=EVAL_TRANSFORM)
        print(f'  → 검증 샘플: {len(val_dataset):,}개')
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=2, pin_memory=True, persistent_workers=True)
    else:
        val_loader = None
        print('  검증 데이터 없음')
else:
    print('실제 데이터 없음 → 더미 데이터 사용')
    X = torch.randn(500, 3, IMG_SIZE, IMG_SIZE)
    ys = torch.randint(0, NUM_STYLES, (500,))
    yc = torch.randint(0, NUM_CATEGORIES, (500,))
    dataset = TensorDataset(X, ys, yc)
    val_loader = None

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=2, pin_memory=True, persistent_workers=True)

K-Fashion 실제 데이터 로드 중...


  캐시 로드: 967,806개
  → 학습 샘플: 967,806개


  캐시 로드: 120,975개
  → 검증 샘플: 120,975개


In [6]:
import warnings

EPOCHS    = 10
model     = StyleCNN(pretrained=True).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
criterion = nn.CrossEntropyLoss()
scaler    = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')

print(f'=== ResNet18 스타일 분류 학습 ({NUM_STYLES} 클래스) ===')
print(f'    배치: {BATCH_SIZE} | Mixed Precision: {device.type == "cuda"}\n')

for epoch in range(1, EPOCHS + 1):
    # 학습
    model.train()
    total_loss, correct = 0, 0
    for batch in loader:
        X, ys, yc = [b.to(device, non_blocking=True) for b in batch]
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            out_s, out_c = model(X)
            loss = criterion(out_s, ys) + 0.3 * criterion(out_c, yc)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        correct    += (out_s.argmax(1) == ys).sum().item()
    train_acc = correct / len(dataset)

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        scheduler.step()

    # 검증
    if val_loader:
        model.eval()
        val_correct = 0
        with torch.no_grad(), torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            for batch in val_loader:
                X, ys, yc = [b.to(device, non_blocking=True) for b in batch]
                out_s, _ = model(X)
                val_correct += (out_s.argmax(1) == ys).sum().item()
        val_acc = val_correct / len(val_dataset)
        print(f'Epoch {epoch:2d}/{EPOCHS} | Loss: {total_loss/len(loader):.4f} | Train: {train_acc:.2%} | Val: {val_acc:.2%} | LR: {scheduler.get_last_lr()[0]:.2e}')
    else:
        print(f'Epoch {epoch:2d}/{EPOCHS} | Loss: {total_loss/len(loader):.4f} | Train: {train_acc:.2%} | LR: {scheduler.get_last_lr()[0]:.2e}')

=== ResNet18 스타일 분류 학습 (24 클래스) ===
    배치: 128 | Mixed Precision: True



Epoch  1/10 | Loss: 1.5785 | Train: 54.81% | Val: 58.82% | LR: 9.76e-05


Epoch  2/10 | Loss: 1.3030 | Train: 61.07% | Val: 62.51% | LR: 9.05e-05


Epoch  3/10 | Loss: 1.1194 | Train: 66.08% | Val: 65.91% | LR: 7.96e-05


Epoch  4/10 | Loss: 0.9655 | Train: 70.63% | Val: 68.14% | LR: 6.58e-05


...

Epoch  6/10 | Loss: 0.7156 | Train: 78.39% | Val: 71.03% | LR: 3.52e-05


Epoch  7/10 | Loss: 0.6102 | Train: 81.76% | Val: 71.87% | LR: 2.14e-05


Epoch  8/10 | Loss: 0.5246 | Train: 84.56% | Val: 72.33% | LR: 1.05e-05


Epoch  9/10 | Loss: 0.4602 | Train: 86.66% | Val: 72.55% | LR: 3.42e-06


Epoch 10/10 | Loss: 0.4219 | Train: 87.94% | Val: 72.71% | LR: 1.00e-06


In [7]:
save_path = r'C:\딥러닝데이터\model_a_cnn.pth'
torch.save(model.state_dict(), save_path)
print(f'저장 완료: {save_path}')

저장 완료: C:\딥러닝데이터\model_a_cnn.pth


In [8]:
# 추론 테스트
model.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    out_s, out_c = model(dummy)
    style_probs = torch.softmax(out_s, dim=1).squeeze().cpu().numpy()
    cat_probs   = torch.softmax(out_c, dim=1).squeeze().cpu().numpy()

print('스타일 예측 상위 5개:')
for s, p in sorted(zip(STYLE_CLASSES, style_probs), key=lambda x: -x[1])[:5]:
    print(f'  {s:12s} {"█" * int(p*20):<20s} {p:.2%}')

top_cat = CATEGORY_CLASSES[cat_probs.argmax()]
print(f'\n대분류 예측: {top_cat} ({cat_probs.max():.2%})')

스타일 예측 상위 5개:


  리조트          █████████            45.24%
  로맨틱          ████                 24.70%
  페미닌          ██                   13.62%
  레트로          ██                   12.84%
  키치                                2.53%

대분류 예측: 원피스 (82.74%)
